# Integrated module workflow

This notebook shows how to intergrate different modules of ChromBERT-tools to infer cell-type-specific enhancer-promoter loop.
- predictive modeling: region_activity_regression
- interpretation: interpret_region_region_interactions




### Download example data

Download example data from Cistrome database:
- HFF_atac_65845.bed: HFF cell-type-specific accessibility peak
- HFF_atac_65845.bw: HFF cell-type-specific accessibility signal
- hESC_atac_104306.bed: hESC cell-type-specific accessibility peak
- hESC_atac_104306.bw: hESC cell-type-specific accessibility signal



In [4]:
import os 
import requests
import pathlib as pt
# def get_bed(name, cell, i):
#     url = 'http://dc2.cistrome.org/api/file?type=bed&id='+str(i)
#     r = requests.get(url, allow_redirects=True)
#     with open(os.path.join( f'{i}_{cell}_{name}.bed'), 'wb') as f: 
#         f.write(r.content)

def get_bed(name, cell, i):
    url = 'http://dc2.cistrome.org/api/file?type=bed&id='+str(i)
    target_file = os.path.join("../data", f"{cell}_{name}_{i}.bed")
    os.system(f"wget '{url}' -O '{target_file}' > wget.out 2>&1")
    
def get_bw(name, cell, i):
    url = 'http://dc2.cistrome.org/api/file?type=bw&id='+str(i)
    target_file = os.path.join("../data", f"{cell}_{name}_{i}.bw")
    os.system(f"wget '{url}' -O '{target_file}' > wget.out 2>&1")
    
if not os.path.exists('../data/HFF_atac_65845.bed'):
    get_bed("atac", "HFF", 65845)
if not os.path.exists('../data/HFF_atac_65845.bw'):
    get_bw("atac", "HFF", 65845)
    
if not os.path.exists('../data/hESC_atac_104306.bed'):
    get_bed("atac", "hESC", 104306)
if not os.path.exists('../data/hESC_atac_104306.bw'):
    get_bw("atac", "hESC", 104306)

### First step: build cell-type-specific model (hESC fine-tuned)

Predict cell-type-specific chromatin accessibility signal

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1" # GPU device

In [6]:
# Predict cell-type-specific chromatin accessibility signal
from chrombert_tools import region_activity_regression
hesc_ft = region_activity_regression(
    acc_peak1="../data/hESC_atac_104306.bed", # cell-type-specific accessibility peak
    acc_signal1="../data/hESC_atac_104306.bw", # cell-type-specific accessibility signal
    odir="./output_hESC", # output directory
)
hesc_ft


Stage 1: prepare chromatin accessibility dataset
Processing stage 1: prepare chromatin accessibility dataset
  Mode: state 1 only (label = log2(1 + cell1 signal) - log2(1 + reference baseline))
[state1_0] Region summary - total: 51067, overlapping with ChromBERT: 51491 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 365
  [state1] merged 1 peak file(s) → 50982 unique ChromBERT regions (after overlap + dedup)
 total region: 50982
  Fast mode: downsampled to ~20000 regions
Finished stage 1
Stage 2: train ChromBERT (state-1 accessibility as log2 signal)

[Attempt 0/2] seed=55
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!


/mnt/Storage/home/chenqianqian/miniconda3/envs/chrombert/lib/python3.9/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `SpearmanCorrcoef` will save all targets and predictions in the buffer. For large datasets, this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [1]
Loading `train_dataloader` to estimate number of stepping batches.
/mnt/Storage/home/chenqianqian/miniconda3/envs/chrombert/l

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/mnt/Storage/home/chenqianqian/miniconda3/envs/chrombert/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:484: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.


Training: |          | 0/? [00:00<?, ?it/s]

...

/mnt/Storage/home/chenqianqian/miniconda3/envs/chrombert/lib/python3.9/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `SpearmanCorrcoef` will save all targets and predictions in the buffer. For large datasets, this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


ft_ckpt: /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/api/output_hESC/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=126.ckpt, test_metrics: {'pearsonr': 0.8317794799804688, 'spearmanr': 0.8037785887718201, 'mse': 0.13317391276359558, 'mae': 0.2869367003440857, 'r2': 0.6761791583964055}
Attempt metrics: pearsonr=0.8317794799804688
Accepted run (pearsonr=0.8318 >= 0.2).

Finished stage 2: obtained a fine-tuned ChromBERT
Best pearsonr=0.8317794799804688, metrics={'pearsonr': 0.8317794799804688, 'spearmanr': 0.8037785887718201, 'mse': 0.13317391276359558, 'mae': 0.2869367003440857, 'r2': 0.6761791583964055, 'ft_ckpt': '/mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/api/output_hESC/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=126.ckpt'}
Finished stage 2 (trained)
Stage 3: Predicting
Region summary - total: 2000, overlapp

Predicting: 100%|██████████| 500/500 [01:42<00:00,  4.88it/s]

  Predictions saved: /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/api/output_hESC/predict/predictions.csv  (2000 regions)
Finished stage 3

All stages completed!
Fine-tuned checkpoint: /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/api/output_hESC/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=126.ckpt
Predictions: ./output_hESC/predict/predictions.csv


ChrombertPredictionRunResult(model=ChromBERTGeneral(
  (pretrain_model): ChromBERTModel(
    (chrombert): ChromBERT(
      (embedding): BERTEmbedding(
        (token): TokenEmbedding(10, 768, padding_idx=0)
        (position): PositionalEmbedding(
          (pe): PositionalEmbeddingTrainable(
            (pe): Embedding(6392, 768)
          )
        )
        (dropout): Dropout(p=0, inplace=False)
      )
      (transformer_blocks): ModuleList(
        (0-7): 8 x EncoderTransformerBlock(
          (attention): SelfAttentionFlashMHA(
            (Wqkv): Linear(in_features=768, out_features=2304, bias=True)
          )
          (feed_forward): PositionwiseFeedForward(
            (w_1): Linear(in_features=768, out_features=3072, bias=True)
            (w_2): Linear(in_features=3072, out_features=768, bias=True)
            (dropout): Dropout(p=0, inplace=False)
            (activation): GELU()
          )
          (input_sublayer): SublayerConnection(
            (norm): LayerNorm()
 

### Second step: interpretation cell-type-specific enhancer-promoter loop (hESC)

In [8]:
hesc_ft_ckpt = hesc_ft.model_ckpt # Fine-tuned cell-type-specific model checkpoint path generated in Step 1
hesc_ft_ckpt

'/mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/api/output_hESC/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=126.ckpt'

In [10]:
# Interpret cell-type-specific enhancer–promoter loops using a fine-tuned checkpoint
# Calculate embedding similarity between enhancers and promoters
# Higher similarity indicates a stronger predicted enhancer–promoter association

from chrombert_tools import interpret_region_region_interactions

hesc_ft_cos = interpret_region_region_interactions(
    region="../data/hESC_atac_104306.bed", # cell-type-specific accessibility peak
    filter_gene_name="ZNF879", # focus on ZNF879 gene promoter
    odir="./output_hESC_sim", # output directory
    ft_ckpt=hesc_ft_ckpt, # fine-tuned cell-type-specific model checkpoint
    distance_min=50000, # minimum distance between enhancer and promoter
    distance_max=250000, # maximum distance between enhancer and promoter
)

Region summary - total: 51067, overlapping with ChromBERT: 51491 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 365
  Gene filter: kept 1/55240 TSS rows (gene_name in [1 names], gene_id in [0 ids])
  Gene filter: kept 3272/51491 region1 (BED) rows on 1 chromosome(s) matching the selected gene(s)
Your supervised_file does not contain the 'label' column. Please verify whether ground truth column ('label') is required. If it is not needed, you may disregard this message.
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!
Loading checkpoint from /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/api/output_hESC/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=1-step=126.ckpt
Loading from pl module, remove prefix 'model.'
Loading from pl modu

100%|██████████| 811/811 [00:56<00:00, 14.39it/s]


Finished!
Enhancer-promoter style pairs saved to: ./output_hESC_sim/tss_region_pairs_cos.tsv


In [11]:
hesc_ft_cos

,chrom,gene_id,gene_name,tss,tss_build_region_index,distal_region_start,distal_region_end,distal_region_build_region_index,dist,dist_bin,cos_sim
0,chr5,ENSG00000234284,ZNF879,179023804,1613833,178896000,178897000,1613738,-126804,-95,0.916769
1,chr5,ENSG00000234284,ZNF879,179023804,1613833,178860000,178861000,1613707,-162804,-126,0.844694
2,chr5,ENSG00000234284,ZNF879,179023804,1613833,179167000,179168000,1613943,143196,110,0.819448
3,chr5,ENSG00000234284,ZNF879,179023804,1613833,178839000,178840000,1613691,-183804,-142,0.698159
4,chr5,ENSG00000234284,ZNF879,179023804,1613833,179261000,179262000,1614021,237196,188,0.644643
5,chr5,ENSG00000234284,ZNF879,179023804,1613833,178793000,178794000,1613653,-229804,-180,0.623318
6,chr5,ENSG00000234284,ZNF879,179023804,1613833,179187000,179188000,1613959,163196,126,0.587667


### Apply the same workflow to another cell type (HFF)

In [13]:
# Build cell-type-specific model
# Predict cell-type-specific chromatin accessibility signal
hFF_ft = region_activity_regression(
    acc_peak1="../data//HFF_atac_65845.bed", # cell-type-specific accessibility peak
    acc_signal1="../data/HFF_atac_65845.bw", # cell-type-specific accessibility signal
    odir="./output_HFF", # output directory
)

hFF_ft_ckpt = hFF_ft.model_ckpt # Fine-tuned cell-type-specific model checkpoint path

# Interpret cell-type-specific enhancer-promoter loop
# Calculate embedding similarity between enhancers and promoters
# Higher similarity indicates a stronger predicted enhancer–promoter association
hFF_ft_cos = interpret_region_region_interactions(
    region="../data/HFF_atac_65845.bed", # cell-type-specific accessibility peak
    filter_gene_name="ZNF879", # focus on ZNF879 gene promoter
    odir="./output_hFF_sim", # output directory
    ft_ckpt=hFF_ft_ckpt, # fine-tuned cell-type-specific model checkpoint
    distance_min=50000, # minimum distance between enhancer and promoter
    distance_max=250000, # maximum distance between enhancer and promoter
)



Stage 1: prepare chromatin accessibility dataset
Processing stage 1: prepare chromatin accessibility dataset
  Mode: state 1 only (label = log2(1 + cell1 signal) - log2(1 + reference baseline))
[state1_0] Region summary - total: 75856, overlapping with ChromBERT: 76027 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 547
  [state1] merged 1 peak file(s) → 75588 unique ChromBERT regions (after overlap + dedup)
 total region: 75588
  Fast mode: downsampled to ~20000 regions
Finished stage 1
Stage 2: train ChromBERT (state-1 accessibility as log2 signal)

[Attempt 0/2] seed=55
Load pretrained ckpt /mnt/Storage/home/chenqianqian/.cache/chrombert/data/checkpoint/hg38_6k_1kb_pretrain.ckpt successfully!


/mnt/Storage/home/chenqianqian/miniconda3/envs/chrombert/lib/python3.9/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `SpearmanCorrcoef` will save all targets and predictions in the buffer. For large datasets, this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [1]
Loading `train_dataloader` to estimate number of stepping batches.
/mnt/Storage/home/chenqianqian/miniconda3/envs/chrombert/lib/python3.9/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:231: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name  | Type             | Params | Mode 
---------------------------------------------------
0 | model | ChromBERTGeneral | 62.8 M |

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/mnt/Storage/home/chenqianqian/miniconda3/envs/chrombert/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:484: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.


Training: |          | 0/? [00:00<?, ?it/s]

...

ft_ckpt: /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/api/output_HFF/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=151.ckpt, test_metrics: {'pearsonr': 0.9385236501693726, 'spearmanr': 0.9424457550048828, 'mse': 0.08400136232376099, 'mae': 0.22429515421390533, 'r2': 0.8802513489479088}
Attempt metrics: pearsonr=0.9385236501693726
Accepted run (pearsonr=0.9385 >= 0.2).

Finished stage 2: obtained a fine-tuned ChromBERT
Best pearsonr=0.9385236501693726, metrics={'pearsonr': 0.9385236501693726, 'spearmanr': 0.9424457550048828, 'mse': 0.08400136232376099, 'mae': 0.22429515421390533, 'r2': 0.8802513489479088, 'ft_ckpt': '/mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/api/output_HFF/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=151.ckpt'}
Finished stage 2 (trained)
Stage 3: Predicting
Region summary - total: 2000, overlapp

Predicting: 100%|██████████| 500/500 [01:41<00:00,  4.91it/s]


  Predictions saved: /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/api/output_HFF/predict/predictions.csv  (2000 regions)
Finished stage 3

All stages completed!
Fine-tuned checkpoint: /mnt/Storage2/home/chenqianqian/projects/chrombert/chrombert_tools/ChromBERT-tools/examples/api/output_HFF/train/try_00_seed_55/lightning_logs/lightning_logs/version_0/checkpoints/epoch=2-step=151.ckpt
Predictions: ./output_HFF/predict/predictions.csv
Region summary - total: 75856, overlapping with ChromBERT: 76027 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 547
  Gene filter: kept 1/55240 TSS rows (gene_name in [1 names], gene_id in [0 ids])
  Gene filter: kept 4610/76027 region1 (BED) rows on 1 chromosome(s) matching the selected gene(s)
Your supervised_file does not contain the 'label' column. Please verify whether ground truth column ('label') is

100%|██████████| 1145/1145 [01:20<00:00, 14.26it/s]


Finished!
Enhancer-promoter style pairs saved to: ./output_hFF_sim/tss_region_pairs_cos.tsv


In [14]:
hFF_ft_cos

,chrom,gene_id,gene_name,tss,tss_build_region_index,distal_region_start,distal_region_end,distal_region_build_region_index,dist,dist_bin,cos_sim
0,chr5,ENSG00000234284,ZNF879,179023804,1613833,178860000,178861000,1613707,-162804,-126,0.880583
1,chr5,ENSG00000234284,ZNF879,179023804,1613833,178896000,178897000,1613738,-126804,-95,0.794756
2,chr5,ENSG00000234284,ZNF879,179023804,1613833,178941000,178942000,1613773,-81804,-60,0.759831
3,chr5,ENSG00000234284,ZNF879,179023804,1613833,179167000,179168000,1613943,143196,110,0.724107
4,chr5,ENSG00000234284,ZNF879,179023804,1613833,179265000,179266000,1614024,241196,191,0.701100
5,chr5,ENSG00000234284,ZNF879,179023804,1613833,179270000,179271000,1614029,246196,196,0.628431
6,chr5,ENSG00000234284,ZNF879,179023804,1613833,179243000,179244000,1614006,219196,173,0.503626
7,chr5,ENSG00000234284,ZNF879,179023804,1613833,179259000,179260000,1614020,235196,187,0.455676
8,chr5,ENSG00000234284,ZNF879,179023804,1613833,179142000,179143000,1613922,118196,89,0.451226
9,chr5,ENSG00000234284,ZNF879,179023804,1613833,179209000,179210000,1613979,185196,146,0.428001


In [15]:
hesc_ft_cos

,chrom,gene_id,gene_name,tss,tss_build_region_index,distal_region_start,distal_region_end,distal_region_build_region_index,dist,dist_bin,cos_sim
0,chr5,ENSG00000234284,ZNF879,179023804,1613833,178896000,178897000,1613738,-126804,-95,0.916769
1,chr5,ENSG00000234284,ZNF879,179023804,1613833,178860000,178861000,1613707,-162804,-126,0.844694
2,chr5,ENSG00000234284,ZNF879,179023804,1613833,179167000,179168000,1613943,143196,110,0.819448
3,chr5,ENSG00000234284,ZNF879,179023804,1613833,178839000,178840000,1613691,-183804,-142,0.698159
4,chr5,ENSG00000234284,ZNF879,179023804,1613833,179261000,179262000,1614021,237196,188,0.644643
5,chr5,ENSG00000234284,ZNF879,179023804,1613833,178793000,178794000,1613653,-229804,-180,0.623318
6,chr5,ENSG00000234284,ZNF879,179023804,1613833,179187000,179188000,1613959,163196,126,0.587667
